In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from scipy.spatial.distance import pdist, squareform

# Add project root to path so we can import src modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.dataset import WiPER81EvalDataset
from src.models import TriNetModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load the Test Dataset (using Gallery for a clean set of identities)
data_dir = os.path.join("..", "data", "Wi-PER81Dataset", "ImageData", "D2t", "Gallery")
dataset = WiPER81EvalDataset(folder_path=data_dir)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

# 2. Initialize the TriNet ConvNeXt-Tiny Model
backbone = 'convnext_tiny'
model = TriNetModel(backbone_name=backbone, embedding_dim=128).to(device)

checkpoint_path = os.path.join("..", "outputs", "checkpoints", f"best_model_trinet_{backbone}.pth")
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

# 3. Extract 128D Embeddings
print(f"Extracting 128D embeddings using {backbone.upper()}...")
all_features = []
all_pids = []

with torch.no_grad():
    for imgs, pids, _ in dataloader:
        imgs = imgs.to(device)
        features = model(imgs)
        all_features.append(features.cpu().numpy())
        all_pids.extend(pids)

embeddings = np.vstack(all_features)
pids = np.array(all_pids)
print(f"Successfully extracted {embeddings.shape[0]} embeddings of shape {embeddings.shape[1]}D.")

In [ ]:
# 1. Select a subset of 15 identities for a readable plot
unique_pids = np.unique(pids)
np.random.seed(42) # For reproducible plots
selected_pids = np.random.choice(unique_pids, size=15, replace=False)

mask = np.isin(pids, selected_pids)
subset_embeddings = embeddings[mask]
subset_pids = pids[mask]

# 2. Run t-SNE Dimensionality Reduction (128D -> 2D)
print("Running t-SNE projection...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
embeddings_2d = tsne.fit_transform(subset_embeddings)

# 3. Plot the Clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=embeddings_2d[:, 0], 
    y=embeddings_2d[:, 1],
    hue=subset_pids,
    palette=sns.color_palette("tab20", n_colors=15),
    legend='full',
    alpha=0.8,
    s=80,
    edgecolor='k'
)

plt.title(f"t-SNE Projection of 128D Embeddings (ConvNeXt-Tiny)\nShowing 15 Identities from D2t Unseen Test Data", 
          fontsize=14, fontweight='bold')
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(title="Person ID (PID)", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
print("Calculating pairwise distance matrix...")
# 1. Compute pairwise Euclidean distances for ALL samples
dist_matrix = squareform(pdist(embeddings, metric='euclidean'))

# 2. Create boolean masks for Genuine (Positive) and Impostor (Negative) pairs
# We use np.triu_indices to only grab the upper triangle, avoiding duplicate pairs and self-comparisons
n = len(pids)
i, j = np.triu_indices(n, k=1)

pairwise_distances = dist_matrix[i, j]
pair_pids_i = pids[i]
pair_pids_j = pids[j]

positive_mask = (pair_pids_i == pair_pids_j)
negative_mask = (pair_pids_i != pair_pids_j)

positive_distances = pairwise_distances[positive_mask]
negative_distances = pairwise_distances[negative_mask]

# 3. Plot the Distributions
plt.figure(figsize=(12, 6))

# Plot Positive (Genuine) pairs
sns.histplot(positive_distances, bins=50, color='tab:green', alpha=0.6, 
             label=f'Positive Pairs (Same ID) - mean: {positive_distances.mean():.2f}', 
             stat='density', kde=True, edgecolor=None)

# Plot Negative (Impostor) pairs
sns.histplot(negative_distances, bins=50, color='tab:red', alpha=0.6, 
             label=f'Negative Pairs (Diff ID) - mean: {negative_distances.mean():.2f}', 
             stat='density', kde=True, edgecolor=None)

# Add the Triplet Loss Margin Line
margin = 0.3
plt.axvline(x=margin, color='black', linestyle='--', linewidth=2, label=f'Triplet Margin ({margin})')

plt.title("Pairwise Distance Distribution (ConvNeXt-Tiny 128D Metric Space)", fontsize=14, fontweight='bold')
plt.xlabel("Euclidean Distance between Embeddings")
plt.ylabel("Density")
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()